# 02 - Intermediate: Recipe Types & Visualizations

This notebook explores py2dataiku's recipe types in depth, demonstrates how to inspect recipe attributes and settings, and covers all available visualization formats and themes.

**Prerequisites:** Completion of `01_beginner.ipynb` or familiarity with `convert()` and `DataikuFlow`.

**What you will learn:**
- All major recipe types and the pandas code that triggers each one
- The `RecipeType` enum and `DataikuRecipe` attributes
- Recipe settings classes (`GroupingSettings`, `JoinSettings`, etc.)
- All 6 visualization formats: SVG, ASCII, HTML, Mermaid, PlantUML, Interactive
- Theme support: `DATAIKU_LIGHT` vs `DATAIKU_DARK`
- Jupyter inline display with `_repr_svg_()`
- YAML round-trip serialization

---
## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

from py2dataiku import (
    convert,
    DataikuFlow,
    DataikuRecipe,
    RecipeType,
    Aggregation,
    JoinKey,
    JoinType,
    AggregationFunction,
    WindowFunctionType,
    SplitMode,
    SamplingMethod,
)
from py2dataiku.models.recipe_settings import (
    RecipeSettings,
    PrepareSettings,
    GroupingSettings,
    JoinSettings,
    WindowSettings,
    SortSettings,
    DistinctSettings,
    TopNSettings,
    PivotSettings,
    SplitSettings,
    SamplingSettings,
    StackSettings,
)
from py2dataiku.visualizers import (
    visualize_flow,
    SVGVisualizer,
    ASCIIVisualizer,
    HTMLVisualizer,
    PlantUMLVisualizer,
    MermaidVisualizer,
    InteractiveVisualizer,
    DataikuTheme,
    DATAIKU_LIGHT,
    DATAIKU_DARK,
)

print("All imports successful.")

---
## 2. The RecipeType Enum

Dataiku DSS has 37 recipe types. py2dataiku models them as the `RecipeType` enum, grouped into visual recipes, code recipes, and ML recipes.

In [ ]:
# List all recipe types
print(f"Total recipe types: {len(RecipeType)}\n")

for rt in RecipeType:
    print(f"  {rt.name:<30s} -> {rt.value!r}")

In [ ]:
# RecipeType enum usage: access by name or value
print("By name:  ", RecipeType.GROUPING)
print("By value: ", RecipeType("grouping"))
print("Value:    ", RecipeType.GROUPING.value)
print("Name:     ", RecipeType.GROUPING.name)

---
## 3. Recipe Types in Action

Each subsection shows the pandas code that maps to a specific Dataiku recipe type, converts it, and inspects the result.

### 3.1 PREPARE Recipe

The Prepare recipe handles column-level transformations like renaming, filling nulls, dropping rows, and type conversions. Most pandas method calls on a single DataFrame map to processors within a Prepare recipe.

In [ ]:
prepare_code = """
import pandas as pd
df = pd.read_csv('sales.csv')
df = df.fillna(0)
df = df.rename(columns={'old_name': 'new_name'})
df = df.dropna()
"""

flow = convert(prepare_code)
print(flow.get_summary())

In [ ]:
# Inspect the Prepare recipe's attributes
for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.PREPARE:
        print(f"Name:        {recipe.name}")
        print(f"Type:        {recipe.recipe_type}")
        print(f"Inputs:      {recipe.inputs}")
        print(f"Outputs:     {recipe.outputs}")
        print(f"Steps:       {len(recipe.steps)}")
        for i, step in enumerate(recipe.steps):
            print(f"  Step {i+1}: {step.processor_type.value} -> {step.get_description()}")
        break

### 3.2 GROUPING Recipe

Pandas `groupby().agg()` maps to a Dataiku Grouping recipe.

In [ ]:
grouping_code = """
import pandas as pd
df = pd.read_csv('orders.csv')
result = df.groupby('category').agg({'amount': 'sum', 'quantity': 'mean'})
"""

flow = convert(grouping_code)
print(flow.get_summary())
print()

# Inspect grouping recipe details
for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.GROUPING:
        print(f"Group keys:    {recipe.group_keys}")
        print(f"Aggregations:  {len(recipe.aggregations)}")
        for agg in recipe.aggregations:
            print(f"  {agg.column} -> {agg.function}")

### 3.3 JOIN Recipe

Pandas `pd.merge()` or `df.merge()` maps to a Join recipe.

In [ ]:
join_code = """
import pandas as pd
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
result = pd.merge(orders, customers, on='customer_id', how='left')
"""

flow = convert(join_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.JOIN:
        print(f"Join type:  {recipe.join_type}")
        print(f"Join keys:  {len(recipe.join_keys)}")
        for key in recipe.join_keys:
            print(f"  Left: {key.left_column}  <->  Right: {key.right_column}")

### 3.4 STACK Recipe

Pandas `pd.concat()` maps to a Stack recipe (union of datasets).

In [ ]:
stack_code = """
import pandas as pd
df1 = pd.read_csv('sales_q1.csv')
df2 = pd.read_csv('sales_q2.csv')
combined = pd.concat([df1, df2])
"""

flow = convert(stack_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.STACK:
        print(f"Inputs:  {recipe.inputs}")
        print(f"Outputs: {recipe.outputs}")

### 3.5 SORT Recipe

Pandas `df.sort_values()` maps to a Sort recipe.

In [ ]:
sort_code = """
import pandas as pd
df = pd.read_csv('products.csv')
df = df.sort_values('price', ascending=False)
"""

flow = convert(sort_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.SORT:
        print(f"Sort columns: {recipe.sort_columns}")

### 3.6 DISTINCT Recipe

Pandas `df.drop_duplicates()` maps to a Distinct recipe.

In [ ]:
distinct_code = """
import pandas as pd
df = pd.read_csv('events.csv')
df = df.drop_duplicates()
"""

flow = convert(distinct_code)
print(flow.get_summary())

### 3.7 TOP_N Recipe

Pandas `df.nlargest()` or `df.nsmallest()` maps to a Top N recipe.

In [ ]:
topn_code = """
import pandas as pd
df = pd.read_csv('scores.csv')
top_10 = df.nlargest(10, 'score')
"""

flow = convert(topn_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.TOP_N:
        print(f"Top N:           {recipe.top_n}")
        print(f"Ranking column:  {recipe.ranking_column}")

### 3.8 WINDOW Recipe

Pandas `df.rolling()` and cumulative functions like `df.cumsum()` map to a Window recipe.

In [ ]:
window_code = """
import pandas as pd
df = pd.read_csv('timeseries.csv')
df['cumulative_sales'] = df['sales'].cumsum()
"""

flow = convert(window_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.WINDOW:
        print(f"Partition columns: {recipe.partition_columns}")
        print(f"Order columns:     {recipe.order_columns}")
        print(f"Aggregations:      {recipe.window_aggregations}")

### 3.9 PIVOT Recipe

Pandas `df.pivot()` or `df.pivot_table()` maps to a Pivot recipe.

In [ ]:
pivot_code = """
import pandas as pd
df = pd.read_csv('sales.csv')
pivoted = df.pivot_table(index='region', columns='product', values='revenue', aggfunc='sum')
"""

flow = convert(pivot_code)
print(flow.get_summary())

### 3.10 SPLIT Recipe

Pandas conditional filtering like `df[condition]` can map to a Split recipe.

In [ ]:
split_code = """
import pandas as pd
df = pd.read_csv('users.csv')
active = df[df['status'] == 'active']
inactive = df[df['status'] != 'active']
"""

flow = convert(split_code)
print(flow.get_summary())

### 3.11 SAMPLING Recipe

Pandas `df.sample()` or `df.head()` maps to a Sampling recipe.

In [ ]:
sampling_code = """
import pandas as pd
df = pd.read_csv('large_dataset.csv')
sample = df.sample(n=1000)
"""

flow = convert(sampling_code)
print(flow.get_summary())
print()

for recipe in flow.recipes:
    if recipe.recipe_type == RecipeType.SAMPLING:
        print(f"Method:      {recipe.sampling_method}")
        print(f"Sample size: {recipe.sample_size}")

### 3.12 New Recipe Handlers: WINDOW, PIVOT, TOP_N, SAMPLING in Detail

The following cells demonstrate that `convert()` now generates dedicated Dataiku visual
recipes for WINDOW, PIVOT, TOP_N, and SAMPLING operations -- previously these fell back
to generic Python recipes.

In [ ]:
# WINDOW recipe from df.rolling().mean() -- now generates a Window recipe (not Python fallback)
window_rolling_code = """
import pandas as pd
df = pd.read_csv('timeseries.csv')
df['rolling_avg'] = df['sales'].rolling(7).mean()
"""

flow = convert(window_rolling_code)
print("Rolling window generates a WINDOW recipe:")
for recipe in flow.recipes:
    print(f"  {recipe.name}: {recipe.recipe_type.value}")
    if recipe.recipe_type == RecipeType.WINDOW:
        print(f"    Partition columns: {recipe.partition_columns}")
        print(f"    Order columns:     {recipe.order_columns}")
        print(f"    Window aggregations: {recipe.window_aggregations}")

In [ ]:
# PIVOT recipe from df.pivot_table() -- now generates a Pivot recipe with PivotSettings
pivot_detail_code = """
import pandas as pd
df = pd.read_csv('quarterly_sales.csv')
pivoted = df.pivot_table(index='region', columns='quarter', values='revenue', aggfunc='sum')
"""

flow = convert(pivot_detail_code)
print("pivot_table() generates a PIVOT recipe:")
for recipe in flow.recipes:
    print(f"  {recipe.name}: {recipe.recipe_type.value}")
    if recipe.recipe_type == RecipeType.PIVOT and recipe.settings:
        print(f"    PivotSettings: {recipe.settings.to_dict()}")

In [ ]:
# TOP_N recipe from df.nlargest() -- now generates a TopN recipe (not Python fallback)
topn_detail_code = """
import pandas as pd
df = pd.read_csv('scores.csv')
top_5 = df.nlargest(5, 'score')
bottom_5 = df.nsmallest(5, 'score')
"""

flow = convert(topn_detail_code)
print("nlargest()/nsmallest() generate TOP_N recipes:")
for recipe in flow.recipes:
    print(f"  {recipe.name}: {recipe.recipe_type.value}")
    if recipe.recipe_type == RecipeType.TOP_N:
        print(f"    top_n={recipe.top_n}, ranking_column={recipe.ranking_column}")

In [ ]:
# SAMPLING recipe from df.sample() -- now generates a Sampling recipe with correct method
sampling_detail_code = """
import pandas as pd
df = pd.read_csv('large_dataset.csv')
fixed_sample = df.sample(n=500)
head_sample = df.head(100)
"""

flow = convert(sampling_detail_code)
print("sample()/head() generate SAMPLING recipes:")
for recipe in flow.recipes:
    print(f"  {recipe.name}: {recipe.recipe_type.value}")
    if recipe.recipe_type == RecipeType.SAMPLING:
        print(f"    method={recipe.sampling_method}, size={recipe.sample_size}")

---
## 4. DataikuRecipe Attributes Deep Dive

Every `DataikuRecipe` has common attributes plus type-specific ones.

In [ ]:
# Build a multi-step flow to inspect recipe attributes
multi_code = """
import pandas as pd
df = pd.read_csv('data.csv')
df = df.fillna(0)
df = df.rename(columns={'old': 'new'})
grouped = df.groupby('category').agg({'value': 'sum'})
sorted_df = grouped.sort_values('value', ascending=False)
"""

flow = convert(multi_code)

for recipe in flow.recipes:
    print(f"--- {recipe} ---")
    print(f"  name:         {recipe.name}")
    print(f"  recipe_type:  {recipe.recipe_type}")
    print(f"  inputs:       {recipe.inputs}")
    print(f"  outputs:      {recipe.outputs}")
    print(f"  source_lines: {recipe.source_lines}")
    print(f"  notes:        {recipe.notes}")
    print(f"  settings:     {recipe.settings}")
    print()

In [ ]:
# The to_dict() method gives a serializable view of a recipe
import json

for recipe in flow.recipes:
    print(f"--- {recipe.name} ---")
    print(json.dumps(recipe.to_dict(), indent=2))
    print()

In [ ]:
# The to_api_dict() method gives the Dataiku API-compatible format
for recipe in flow.recipes:
    print(f"--- {recipe.name} (API format) ---")
    print(json.dumps(recipe.to_api_dict(), indent=2))
    print()

### 4.1 Factory Methods

`DataikuRecipe` provides factory methods for common recipe types.

In [ ]:
# Create recipes using factory methods
grouping = DataikuRecipe.create_grouping(
    name="compute_group_stats",
    input_dataset="cleaned_data",
    output_dataset="grouped_stats",
    keys=["region", "product"],
    aggregations=[
        Aggregation(column="revenue", function="SUM"),
        Aggregation(column="quantity", function="AVG"),
    ],
)
print(grouping)
print(f"Group keys:    {grouping.group_keys}")
print(f"Aggregations:  {[(a.column, a.function) for a in grouping.aggregations]}")

In [ ]:
# Factory: create_join
join_recipe = DataikuRecipe.create_join(
    name="join_orders_customers",
    left_dataset="orders",
    right_dataset="customers",
    output_dataset="orders_enriched",
    join_keys=[JoinKey(left_column="cust_id", right_column="id")],
    join_type=JoinType.LEFT,
)
print(join_recipe)
print(f"Join type: {join_recipe.join_type}")
print(f"Keys:      {[(k.left_column, k.right_column) for k in join_recipe.join_keys]}")

### 4.2 Helper Methods on DataikuRecipe

In [ ]:
# add_aggregation and add_note
grouping.add_aggregation("revenue", "MAX", output_column="max_revenue")
grouping.add_note("Added max revenue aggregation")

print(f"Aggregations now: {len(grouping.aggregations)}")
for a in grouping.aggregations:
    print(f"  {a.column} -> {a.function} (output: {a.output_column})")

print(f"\nNotes: {grouping.notes}")

In [ ]:
# add_join_key
join_recipe.add_join_key("region_code", "region_code")
print(f"Join keys: {len(join_recipe.join_keys)}")
for k in join_recipe.join_keys:
    print(f"  {k.left_column} <-> {k.right_column} ({k.match_type})")

---
## 5. Recipe Settings Classes

Recipe settings use a composition pattern. Each recipe type has its own settings dataclass that can be attached via `recipe.settings`.

In [ ]:
# GroupingSettings
gs = GroupingSettings(
    keys=["department", "year"],
    aggregations=[
        Aggregation(column="salary", function="AVG"),
        Aggregation(column="salary", function="MAX"),
    ],
    global_count=True,
)
print("GroupingSettings.to_dict():")
print(json.dumps(gs.to_dict(), indent=2))
print("\nGroupingSettings.to_display_dict():")
print(json.dumps(gs.to_display_dict(), indent=2))

In [ ]:
# JoinSettings
js = JoinSettings(
    join_type="INNER",
    join_keys=[JoinKey(left_column="id", right_column="user_id")],
    selected_columns={"left": ["id", "name"], "right": ["email"]},
)
print("JoinSettings.to_dict():")
print(json.dumps(js.to_dict(), indent=2))

In [ ]:
# WindowSettings
ws = WindowSettings(
    partition_columns=["department"],
    order_columns=["hire_date"],
    aggregations=[
        {"column": "salary", "type": WindowFunctionType.RUNNING_SUM},
        {"column": "employee_id", "type": WindowFunctionType.ROW_NUMBER},
    ],
)
print("WindowSettings.to_dict():")
# WindowFunctionType enums are serialized to their string values
print(json.dumps(ws.to_dict(), indent=2))

In [ ]:
# SortSettings
ss = SortSettings(
    sort_columns=[
        {"column": "revenue", "order": "DESC"},
        {"column": "name", "order": "ASC"},
    ]
)
print("SortSettings.to_dict():")
print(json.dumps(ss.to_dict(), indent=2))

In [ ]:
# PivotSettings
ps = PivotSettings(
    row_columns=["region"],
    column_column="quarter",
    value_column="revenue",
    aggregation="SUM",
)
print("PivotSettings.to_dict():")
print(json.dumps(ps.to_dict(), indent=2))

In [ ]:
# Other settings: SplitSettings, SamplingSettings, DistinctSettings, TopNSettings, StackSettings
for SettingsClass, kwargs in [
    (SplitSettings, {"split_mode": "FILTER", "condition": "age > 18"}),
    (SamplingSettings, {"sampling_method": "RANDOM", "sample_size": 500}),
    (DistinctSettings, {"compute_count": True}),
    (TopNSettings, {"top_n": 20, "ranking_column": "score"}),
    (StackSettings, {"mode": "UNION"}),
]:
    settings = SettingsClass(**kwargs)
    print(f"{SettingsClass.__name__}: {settings.to_dict()}")

In [ ]:
# Attaching settings to a recipe (composition pattern)
recipe = DataikuRecipe(
    name="compute_pivot",
    recipe_type=RecipeType.PIVOT,
    inputs=["sales_data"],
    outputs=["pivoted_sales"],
    settings=PivotSettings(
        row_columns=["region", "product"],
        column_column="month",
        value_column="amount",
        aggregation="SUM",
    ),
)

# Settings take precedence over flat fields in to_dict() and to_api_dict()
print("Recipe to_dict():")
print(json.dumps(recipe.to_dict(), indent=2))
print("\nRecipe to_api_dict():")
print(json.dumps(recipe.to_api_dict(), indent=2))

---
## 6. Supporting Enums

py2dataiku provides enums for join types, aggregation functions, window functions, split modes, and sampling methods.

In [ ]:
# JoinType
print("JoinType values:")
for jt in JoinType:
    print(f"  {jt.name} = {jt.value!r}")

print("\nAggregationFunction values:")
for af in AggregationFunction:
    print(f"  {af.name} = {af.value!r}")

In [ ]:
# WindowFunctionType
print("WindowFunctionType values:")
for wf in WindowFunctionType:
    print(f"  {wf.name} = {wf.value!r}")

In [ ]:
# SplitMode and SamplingMethod
print("SplitMode values:")
for sm in SplitMode:
    print(f"  {sm.name} = {sm.value!r}")

print("\nSamplingMethod values:")
for sm in SamplingMethod:
    print(f"  {sm.name} = {sm.value!r}")

---
## 7. Visualization Formats

py2dataiku supports 6 visualization formats. All are accessed through `flow.visualize(format=...)` or directly via `visualize_flow(flow, format=...)`.

We will use a multi-step flow as our visualization subject.

In [ ]:
# Build a representative flow for visualization
viz_code = """
import pandas as pd

# Load data
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')

# Join
merged = pd.merge(orders, customers, on='customer_id', how='left')

# Clean
merged = merged.fillna(0)
merged = merged.dropna()

# Aggregate
summary = merged.groupby('region').agg({'revenue': 'sum', 'quantity': 'mean'})

# Sort
summary = summary.sort_values('revenue', ascending=False)
"""

viz_flow = convert(viz_code)
print(viz_flow.get_summary())

### 7.1 SVG Visualization (Default)

Pixel-accurate representation matching the Dataiku DSS interface.

In [ ]:
# SVG output (renders inline in Jupyter)
from IPython.display import SVG, display

svg_content = viz_flow.visualize(format="svg")
display(SVG(svg_content))

### 7.2 ASCII Visualization

Terminal-friendly text art. Great for logging and console output.

In [ ]:
ascii_output = viz_flow.visualize(format="ascii")
print(ascii_output)

### 7.3 HTML Visualization

Interactive canvas-based visualization.

In [ ]:
from IPython.display import HTML

html_content = viz_flow.visualize(format="html")
print(f"HTML output length: {len(html_content)} characters")
print(f"First 200 chars: {html_content[:200]}...")

### 7.4 Mermaid Visualization

Mermaid diagrams are compatible with GitHub, Notion, and many documentation tools.

In [ ]:
mermaid_output = viz_flow.visualize(format="mermaid")
print(mermaid_output)

### 7.5 PlantUML Visualization

PlantUML diagrams for documentation and technical specs.

In [ ]:
plantuml_output = viz_flow.visualize(format="plantuml")
print(plantuml_output)

### 7.6 Interactive Visualization

Enhanced HTML with pan/zoom, search, and export capabilities.

In [ ]:
interactive_content = viz_flow.visualize(format="interactive")
print(f"Interactive output length: {len(interactive_content)} characters")
print(f"First 200 chars: {interactive_content[:200]}...")

### 7.7 Using `visualize_flow()` Directly

The module-level function `visualize_flow()` provides the same functionality.

In [ ]:
# Same result, using the module-level function
ascii_v2 = visualize_flow(viz_flow, format="ascii")
print(ascii_v2)

### 7.8 Convenience Methods on DataikuFlow

`DataikuFlow` provides shortcut methods for each format.

In [ ]:
# Direct methods on flow
svg_out = viz_flow.to_svg()       # returns SVG string
ascii_out = viz_flow.to_ascii()   # returns ASCII string
html_out = viz_flow.to_html()     # returns HTML string
puml_out = viz_flow.to_plantuml() # returns PlantUML string

print(f"to_svg():      {len(svg_out)} chars")
print(f"to_ascii():    {len(ascii_out)} chars")
print(f"to_html():     {len(html_out)} chars")
print(f"to_plantuml(): {len(puml_out)} chars")

---
## 8. Theme Support

py2dataiku provides two built-in themes: `DATAIKU_LIGHT` (default) and `DATAIKU_DARK`. You can also create custom themes.

In [ ]:
# Inspect the light theme
print(f"Theme: {DATAIKU_LIGHT.name}")
print(f"Background:   {DATAIKU_LIGHT.background_color}")
print(f"Font family:  {DATAIKU_LIGHT.font_family}")
print(f"Dataset size: {DATAIKU_LIGHT.dataset_width}x{DATAIKU_LIGHT.dataset_height}")
print(f"Recipe size:  {DATAIKU_LIGHT.recipe_size}")
print(f"\nRecipe colors (bg, border, text):")
for name, colors in DATAIKU_LIGHT.recipe_colors.items():
    print(f"  {name:<12s}: {colors}")

In [ ]:
# Inspect the dark theme
print(f"Theme: {DATAIKU_DARK.name}")
print(f"Background:  {DATAIKU_DARK.background_color}")
print(f"Input bg:    {DATAIKU_DARK.input_bg}")
print(f"Output bg:   {DATAIKU_DARK.output_bg}")

In [ ]:
# Render with DATAIKU_LIGHT theme (default)
light_svg = SVGVisualizer(theme=DATAIKU_LIGHT).render(viz_flow)
display(SVG(light_svg))

In [ ]:
# Render with DATAIKU_DARK theme
dark_svg = SVGVisualizer(theme=DATAIKU_DARK).render(viz_flow)
display(SVG(dark_svg))

In [ ]:
# You can also pass the theme via visualize_flow()
dark_ascii = ASCIIVisualizer(theme=DATAIKU_DARK).render(viz_flow)
print(dark_ascii)

In [ ]:
# Custom theme: create your own DataikuTheme
custom_theme = DataikuTheme(
    name="custom-blue",
    background_color="#F0F4FF",
    input_bg="#DBEAFE",
    input_border="#3B82F6",
    input_text="#1E40AF",
    output_bg="#D1FAE5",
    output_border="#10B981",
    output_text="#065F46",
    dataset_width=180,
    recipe_size=80,
)
custom_svg = SVGVisualizer(theme=custom_theme).render(viz_flow)
display(SVG(custom_svg))

---
## 9. Jupyter Inline Display with `_repr_svg_()`

When you evaluate a `DataikuFlow` as the last expression in a Jupyter cell, it automatically renders as SVG via the `_repr_svg_()` method.

In [ ]:
# Simply evaluate the flow -- Jupyter renders it as SVG automatically
viz_flow

---
## 10. YAML Serialization

`DataikuFlow` supports round-trip YAML serialization via `to_yaml()` and `from_yaml()`.

In [ ]:
# Serialize flow to YAML
yaml_str = viz_flow.to_yaml()
print(yaml_str)

In [ ]:
# Round-trip: reconstruct from YAML
restored_flow = DataikuFlow.from_yaml(yaml_str)
print(f"Original:  {viz_flow}")
print(f"Restored:  {restored_flow}")
print(f"\nRecipes match: {len(viz_flow.recipes) == len(restored_flow.recipes)}")
print(f"Datasets match: {len(viz_flow.datasets) == len(restored_flow.datasets)}")

In [ ]:
# JSON round-trip also works
json_str = viz_flow.to_json(indent=2)
restored_json = DataikuFlow.from_json(json_str)
print(f"JSON round-trip: {len(restored_json.recipes)} recipes, {len(restored_json.datasets)} datasets")

In [ ]:
# Verify the restored flow visualizes correctly
restored_flow

---
## 11. Flow Querying Methods

`DataikuFlow` provides methods for querying recipes and datasets.

In [ ]:
# Iteration protocol: len() and for-loop
print(f"Number of recipes: {len(viz_flow)}")
print()

for recipe in viz_flow:
    print(f"  {recipe}")

In [ ]:
# Query by type
grouping_recipes = viz_flow.get_recipes_by_type(RecipeType.GROUPING)
print(f"Grouping recipes: {grouping_recipes}")

prepare_recipes = viz_flow.get_recipes_by_type(RecipeType.PREPARE)
print(f"Prepare recipes:  {prepare_recipes}")

In [ ]:
# Dataset classification
print(f"Input datasets:        {[ds.name for ds in viz_flow.input_datasets]}")
print(f"Intermediate datasets: {[ds.name for ds in viz_flow.intermediate_datasets]}")
print(f"Output datasets:       {[ds.name for ds in viz_flow.output_datasets]}")

---
## 12. DAG Analysis

The `flow.graph` property provides a `FlowGraph` for DAG analysis.
Let's explore the structure of our `viz_flow` from section 7.

In [ ]:
# Access the DAG from the flow we built in section 7
dag = viz_flow.graph
print(f"Flow graph: {dag}")
print(f"Dataset nodes: {len(dag.dataset_nodes)}")
print(f"Recipe nodes: {len(dag.recipe_nodes)}")

# Topological sort -- the execution order
print("\nExecution order (topological sort):")
for i, name in enumerate(dag.topological_sort(), 1):
    node = dag.get_node(name)
    print(f"  {i}. [{node.node_type.value}] {name}")

# Roots and leaves
print(f"\nRoot nodes (sources): {dag.get_roots()}")
print(f"Leaf nodes (sinks): {dag.get_leaves()}")

# Cycle detection and connected components
print(f"Cycles: {dag.detect_cycles()}")
print(f"Connected components: {len(dag.find_disconnected_subgraphs())}")

---
## 13. Column Lineage

The `get_column_lineage()` method traces a column through the flow's recipes
to discover its origin and transformations.

In [ ]:
# Trace the 'revenue' column through the viz_flow
lineage = viz_flow.get_column_lineage("revenue")
print(f"Column: {lineage.column}")
print(f"Final dataset: {lineage.final_dataset}")
print(f"Origin dataset: {lineage.origin_dataset}")
print(f"Origin column: {lineage.origin_column}")
print(f"Transformations: {lineage.transformations}")

# Also trace 'quantity'
lineage_qty = viz_flow.get_column_lineage("quantity")
print(f"\nColumn 'quantity':")
print(f"  Origin: {lineage_qty.origin_dataset} -> Final: {lineage_qty.final_dataset}")
print(f"  Transformations: {lineage_qty.transformations}")

---
## 14. Format Comparison Guide

| Format | Best For | Rendering | Interactivity |
|--------|----------|-----------|---------------|
| **ASCII** | Terminal output, logs, CI/CD | Plain text | None |
| **SVG** | Documentation, Jupyter inline | Browser/viewer | None |
| **HTML** | Web dashboards, sharing | Browser | Basic |
| **Mermaid** | GitHub READMEs, Notion, Markdown | Mermaid renderer | None |
| **PlantUML** | Technical docs, UML tools | PlantUML renderer | None |
| **Interactive** | Exploration, large flows | Browser | Pan/zoom/search |

In [ ]:
# Compare output sizes across all formats
print(f"{'Format':<15s} {'Output Size':>12s}")
print("-" * 28)
for fmt in ["ascii", "svg", "html", "mermaid", "plantuml", "interactive"]:
    output = viz_flow.visualize(format=fmt)
    print(f"{fmt:<15s} {len(output):>10,d} chars")

---
## Summary

In this notebook we covered:

1. **RecipeType enum** -- All 37 recipe types organized by category (visual, code, ML)
2. **Recipe conversions** -- Pandas code that maps to PREPARE, GROUPING, JOIN, STACK, SORT, DISTINCT, TOP_N, WINDOW, PIVOT, SPLIT, and SAMPLING recipes
3. **DataikuRecipe attributes** -- `name`, `recipe_type`, `inputs`, `outputs`, `settings`, `steps`, `group_keys`, `aggregations`, etc.
4. **Factory methods** -- `create_grouping()`, `create_join()`, `create_prepare()`, `create_python()`
5. **Recipe settings classes** -- `GroupingSettings`, `JoinSettings`, `WindowSettings`, `SortSettings`, `PivotSettings`, `SplitSettings`, `SamplingSettings`, `DistinctSettings`, `TopNSettings`, `StackSettings`
6. **Visualization formats** -- SVG, ASCII, HTML, Mermaid, PlantUML, Interactive
7. **Theme support** -- `DATAIKU_LIGHT`, `DATAIKU_DARK`, and custom `DataikuTheme`
8. **Jupyter integration** -- `_repr_svg_()` for automatic inline rendering
9. **YAML/JSON serialization** -- `to_yaml()`/`from_yaml()`, `to_json()`/`from_json()` round-trips

**Next:** Continue to `03_advanced.ipynb` to explore processors, DAG analysis, and flow optimization.